In [0]:
%sql
CREATE WIDGET TEXT run_month DEFAULT '2026-01';

In [0]:
%sql
USE com_edp_prd.cmpa_insights_internal_schema;

In [0]:
%sql
CREATE OR REPLACE TABLE payer360_dq_results (
  run_month STRING,
  check_category STRING,
  check_name STRING,
  status STRING,
  failed_row_count BIGINT,
  total_row_count BIGINT,
  failure_threshold STRING,
  comments STRING,
  run_timestamp TIMESTAMP
)
USING DELTA;


In [0]:
%sql
INSERT INTO payer360_dq_results
SELECT
  '${run_month}',
  'SNAPSHOT',
  'BASE_ROW_COUNT',
  'INFO',
  0,
  COUNT(*),
  'Reference only',
  'Total payer-plan rows for the month',
  current_timestamp()
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master;


In [0]:
%sql
INSERT INTO payer360_dq_results
SELECT
  '${run_month}',
  'KEY_INTEGRITY',
  'DUPLICATE_PAYER_PLAN',
  CASE WHEN COUNT(*) > 0 THEN 'FAIL' ELSE 'PASS' END,
  COUNT(*),
  (SELECT COUNT(*) FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master),
  'Must be zero',
  'Duplicate PAYER_ID + PLAN_ID rows',
  current_timestamp()
FROM (
  SELECT PAYER_ID, PLAN_ID, PHARMACY_CHANNEL
  FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
  GROUP BY PAYER_ID, PLAN_ID, PHARMACY_CHANNEL
  HAVING COUNT(*) > 1
);


In [0]:
%sql
INSERT INTO payer360_dq_results
SELECT
  '${run_month}',
  'COMPLETENESS',
  'MISSING_PAYER_OR_PLAN_ID',
  CASE WHEN COUNT(*) > 0 THEN 'FAIL' ELSE 'PASS' END,
  COUNT(*),
  (SELECT COUNT(*) FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master),
  'Must be zero',
  'PAYER_ID or PLAN_ID is NULL',
  current_timestamp()
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
WHERE PAYER_ID IS NULL
   OR PLAN_ID IS NULL;


In [0]:
%sql
INSERT INTO payer360_dq_results
SELECT
  '${run_month}',
  'BUSINESS_LOGIC',
  'ELAPRASE_GT_TOTAL_LIVES',
  CASE WHEN COUNT(*) > 0 THEN 'FAIL' ELSE 'PASS' END,
  COUNT(*),
  (SELECT COUNT(*) FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master),
  'Must be zero',
  'Total Elaprase patients exceed total lives',
  current_timestamp()
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
WHERE total_elaprase_patients > total_lives;


In [0]:
%sql
INSERT INTO payer360_dq_results
SELECT
  '${run_month}',
  'BUSINESS_LOGIC',
  'NEW_PATIENTS_GT_TOTAL_ELAPRASE',
  CASE WHEN COUNT(*) > 0 THEN 'FAIL' ELSE 'PASS' END,
  COUNT(*),
  (SELECT COUNT(*) FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master),
  'Must be zero',
  'New Elaprase patients exceed total Elaprase patients',
  current_timestamp()
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
WHERE new_elaprase_patients_qtr > total_elaprase_patients
   OR new_elaprase_patients_mth > total_elaprase_patients;


In [0]:
%sql
INSERT INTO payer360_dq_results
SELECT
  '${run_month}',
  'TEMPORAL_LOGIC',
  'MONTHLY_GT_QUARTERLY_NEW',
  CASE WHEN COUNT(*) > 0 THEN 'WARN' ELSE 'PASS' END,
  COUNT(*),
  (SELECT COUNT(*) FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master),
  'Expected zero',
  'Monthly new patients exceed quarterly new patients',
  current_timestamp()
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
WHERE new_elaprase_patients_mth > new_elaprase_patients_qtr;


In [0]:
%sql
INSERT INTO payer360_dq_results
SELECT
  '${run_month}',
  'CLAIMS_INTEGRITY',
  'CLAIM_COMPONENTS_GT_TOTAL',
  CASE WHEN COUNT(*) > 0 THEN 'FAIL' ELSE 'PASS' END,
  COUNT(*),
  (SELECT COUNT(*) FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master),
  'Must be zero',
  'Approved + Rejected + Reversed exceeds total claims',
  current_timestamp()
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
WHERE approved_fills + rejected_fills + reversed_fills > total_claims;


In [0]:
%sql
INSERT INTO payer360_dq_results
SELECT
  '${run_month}',
  'METRIC_VALIDATION',
  'REJECTION_RATE_OUT_OF_BOUNDS',
  CASE WHEN COUNT(*) > 0 THEN 'FAIL' ELSE 'PASS' END,
  COUNT(*),
  (SELECT COUNT(*) FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master),
  'Must be between 0 and 1',
  'Elaprase rejection rate < 0 or > 1',
  current_timestamp()
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
WHERE elaprase_rejection_rate < 0
   OR elaprase_rejection_rate > 1;


In [0]:
%sql
INSERT INTO payer360_dq_results
SELECT
  '${run_month}',
  'RANKING',
  'MISSING_PAYER_RANK',
  CASE WHEN COUNT(*) > 0 THEN 'WARN' ELSE 'PASS' END,
  COUNT(*),
  (SELECT COUNT(*) FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master),
  'Expected zero',
  'Payer rank missing where total lives > 0',
  current_timestamp()
FROM com_edp_prd.cmpa_insights_internal_schema.payer360_master
WHERE total_lives > 0
  AND (payer_rank IS NULL OR payer_rank = 0);


In [0]:
%sql
SELECT * FROM payer360_dq_results;